# Etapa 3 — Probabilidade

**Pergunta prática:** qual é a probabilidade de hospitalização entre os pacientes notificados com dengue e como ela se relaciona com leucopenia e vômito?

**Dados:** Sinan/Dengue 2026 (`DENGBR26.csv`). Usamos os registros com hospitalização conhecida, sem balanceamento, para manter as proporções observadas.

No Google Colab, envie `DENGBR26.csv` para a pasta `/content` e use **Ambiente de execução → Executar tudo**. Este notebook não depende do `utils.py` das outras etapas.

## 1. Importação e leitura dos dados

In [1]:
import pandas as pd

CAMINHO_CSV = "/content/DENGBR26.csv"

dados = pd.read_csv(
    CAMINHO_CSV,
    encoding="latin-1",
    usecols=["HOSPITALIZ", "LEUCOPENIA", "VOMITO"],
    low_memory=False
)

dados = dados[dados["HOSPITALIZ"].isin([1, 2])].copy()

print("Registros utilizados:", len(dados))

Registros utilizados: 346765


## 2. Distribuição conjunta

Usamos `1 = sim` e `2 = não`. A tabela apresenta as quatro combinações de hospitalização e leucopenia.

In [2]:
a = dados[dados["LEUCOPENIA"].isin([1, 2])].copy()
tabela = pd.crosstab(a["HOSPITALIZ"], a["LEUCOPENIA"], normalize="all")
print(tabela.round(4))
print("Soma:", tabela.to_numpy().sum())

LEUCOPENIA     1.0     2.0
HOSPITALIZ                
1.0         0.0131  0.0725
2.0         0.0336  0.8808
Soma: 1.0


## 3. Probabilidades marginais e condicionais

In [3]:
h = a["HOSPITALIZ"] == 1
l = a["LEUCOPENIA"] == 1

p_h = h.mean()
p_l = l.mean()
p_h_l = h[l].mean()
p_h_sem_l = h[~l].mean()

print("P(Hospitalização):", round(p_h, 4))
print("P(Leucopenia):", round(p_l, 4))
print("P(Hospitalização | Leucopenia):", round(p_h_l, 4))
print("P(Hospitalização | Sem leucopenia):", round(p_h_sem_l, 4))

P(Hospitalização): 0.0856
P(Leucopenia): 0.0467
P(Hospitalização | Leucopenia): 0.2808
P(Hospitalização | Sem leucopenia): 0.076


A probabilidade de hospitalização entre os pacientes com leucopenia é maior que entre aqueles sem leucopenia. Isso mostra associação nos registros, não uma relação de causa e efeito.

## 4. Regra da cadeia

Calculamos $P(H,L,V) = P(H)\,P(L|H)\,P(V|H,L)$ e comparamos com a frequência direta.

In [4]:
b = dados[dados["LEUCOPENIA"].isin([1, 2]) & dados["VOMITO"].isin([1, 2])].copy()
h = b["HOSPITALIZ"] == 1
l = b["LEUCOPENIA"] == 1
v = b["VOMITO"] == 1

p_h = h.mean()
p_l_h = l[h].mean()
p_v_h_l = v[h & l].mean()
p_cadeia = p_h * p_l_h * p_v_h_l
p_direta = (h & l & v).mean()

print("P(H):", round(p_h, 4))
print("P(L|H):", round(p_l_h, 4))
print("P(V|H,L):", round(p_v_h_l, 4))
print("Regra da cadeia:", round(p_cadeia, 4))
print("Probabilidade direta:", round(p_direta, 4))

P(H): 0.0856
P(L|H): 0.1533
P(V|H,L): 0.4732
Regra da cadeia: 0.0062
Probabilidade direta: 0.0062


## 5. Teorema de Bayes

Calculamos $P(H|L) = P(L|H)\,P(H)/P(L)$ e verificamos o resultado diretamente.

In [5]:
p_l = l.mean()
p_h_l_bayes = p_l_h * p_h / p_l
p_h_l_direta = h[l].mean()

print("P(H):", round(p_h, 4))
print("P(L):", round(p_l, 4))
print("P(L|H):", round(p_l_h, 4))
print("P(H|L) pelo Teorema de Bayes:", round(p_h_l_bayes, 4))
print("P(H|L) diretamente:", round(p_h_l_direta, 4))

P(H): 0.0856
P(L): 0.0467
P(L|H): 0.1533
P(H|L) pelo Teorema de Bayes: 0.2808
P(H|L) diretamente: 0.2808


$P(L|H)$ descreve a frequência de leucopenia entre os hospitalizados (direção do desfecho para a evidência). Já $P(H|L)$ descreve a frequência de hospitalização entre pacientes com leucopenia (interpretação diagnóstica). Nenhuma dessas probabilidades comprova causalidade.

## 6. Independência condicional

Verificamos se $P(L,V|H)$ é igual a $P(L|H)P(V|H)$.

In [6]:
p_l_h = l[h].mean()
p_v_h = v[h].mean()
p_lv_h = (l & v)[h].mean()
produto = p_l_h * p_v_h

print("P(L|H):", round(p_l_h, 4))
print("P(V|H):", round(p_v_h, 4))
print("P(L,V|H):", round(p_lv_h, 4))
print("Produto:", round(produto, 4))
print("Diferença:", round(abs(p_lv_h - produto), 4))

P(L|H): 0.1533
P(V|H): 0.4202
P(L,V|H): 0.0725
Produto: 0.0644
Diferença: 0.0081


Como os dois valores não são iguais, leucopenia e vômito não são exatamente independentes entre os hospitalizados nos dados observados. A diferença é pequena, mas não é zero.

## 7. Limitações da lógica

A lógica de primeira ordem pode representar regras sobre sintomas e hospitalização, mas não expressa diretamente graus de incerteza. Por exemplo, uma regra lógica sozinha não informa que um paciente tem 70% de chance de ser hospitalizado. As probabilidades permitem representar esse tipo de incerteza, porque pacientes com sintomas semelhantes podem ter desfechos diferentes.